# T01 — 첫 MCP 서버 만들기 (Tutorial)

> 🎓 **학생 친화 튜토리얼** — 친숙한 도메인(시간/덧셈/면적)으로 MCP 서버 기초를 학습합니다.

## 학습 목표
1. MCP의 3대 핵심 기능(Tools / Resources / Prompts) 개념 이해
2. `FastMCP`로 서버 인스턴스 생성
3. `@mcp.tool()` 데코레이터로 3가지 도구(시간, 덧셈, 면적) 정의
4. 자동 스키마 생성 확인 + 도구 직접 호출 테스트
5. `tutorial_server.py` 파일로 저장 (T02·T03 의존)

## Prerequisites
- Python 3.11 이상
- 약 30분 소요
- Jupyter Notebook 또는 VS Code (Python extension)

> 📖 **강의노트 매핑**: `Week_07.md §1.1` (MCP 서버 기초), `§1.4` (도구 정의)와 같은 흐름. 단 본 튜토리얼은 친숙한 도메인(시간/덧셈/면적)으로 진행합니다. Skilljar 원본 `DocumentMCP` 예제는 `skilljar/S6_01_mcp_server.ipynb`에서 학습하세요.


## §0. 환경 준비

먼저 MCP SDK와 의존성 패키지를 확인합니다. 아래 셀을 **무조건 먼저 실행**하세요.

In [ ]:
# Setup
# 만약 import 에러가 발생하면 다음 명령으로 설치하세요:
#   pip install "mcp[cli]" pydantic
#
# 한 줄로 설치:
#   !pip install "mcp[cli]" pydantic

import json
from datetime import datetime
from mcp.server.fastmcp import FastMCP

print("OK — MCP SDK가 정상적으로 import되었습니다.")


## §1. MCP가 무엇인가? — Tool Use vs MCP

Week 04에서 배운 Tool Use는 도구 스키마와 실행 로직이 **앱 코드에 내장**되어 있었습니다. 이러면 같은 도구를 여러 앱에서 쓸 때마다 코드를 복사해야 합니다.

MCP(Model Context Protocol)는 도구를 **독립된 서버**로 분리하여:
- 어떤 LLM 호스트(Claude Desktop, Claude Code, 커스텀 앱)든 같은 프로토콜로 연결
- 한 번 만들어 여러 곳에서 재사용
- Tools + Resources + Prompts 3가지 기능 제공

| 항목 | Tool Use (W4) | MCP (W7) |
|------|--------------|----------|
| 도구 정의 위치 | 앱 코드 안에 JSON Schema | MCP 서버에 `@mcp.tool()` |
| 스키마 생성 | 수동 | 자동 (타입 힌트 + docstring) |
| 재사용성 | 해당 앱에서만 | 어떤 MCP 클라이언트든 |
| 추가 기능 | 도구만 | Tools + Resources + Prompts |

> 💡 **핵심 비유**: MCP는 "AI를 위한 USB-C"입니다. 한 번 만들어둔 어댑터(서버)를 어떤 노트북(LLM 호스트)에도 꽂을 수 있습니다.


## §2. 빈 서버 만들기

먼저 도구가 없는 빈 MCP 서버를 만들어 봅니다. `FastMCP("이름")`으로 인스턴스를 생성합니다.

In [ ]:
# 빈 MCP 서버 인스턴스 생성
mcp = FastMCP("My First MCP Server")

print(f"서버 이름: {mcp.name}")
print("서버가 생성되었습니다. 아직 도구가 없으므로 기능은 비어 있습니다.")


> ☑ **체크포인트 1**: 위 셀을 실행하면 `서버 이름: My First MCP Server`가 출력되어야 합니다.
>
> ❌ 만약 안 나온다면:
> - `from mcp.server.fastmcp import FastMCP` import 에러 → `pip install "mcp[cli]"`
> - Python 버전 확인 → `python --version` (3.11 이상 필요)


## §3. 첫 도구 — 현재 시간

`@mcp.tool()` 데코레이터로 함수를 도구로 등록합니다. 타입 힌트와 docstring만 작성하면 **JSON Schema가 자동 생성**됩니다.

> 💡 Week 04에서는 JSON Schema를 손으로 작성했지만, FastMCP는 함수 시그니처에서 자동 추출합니다.

In [ ]:
# 도구 1: 현재 시간 조회
@mcp.tool()
def get_current_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """현재 날짜와 시간을 반환합니다.

    Args:
        format: 날짜/시간 형식 (Python strftime 포맷)
    """
    return datetime.now().strftime(format)


print("get_current_time 도구가 등록되었습니다.")


## §4. 두 번째 도구 — 덧셈

숫자를 더하는 단순한 도구입니다. 인자 타입(`float`)과 반환 타입(`float`)을 지정하면 스키마가 자동 추출됩니다.

In [ ]:
# 도구 2: 두 숫자 더하기
@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    """두 숫자를 더합니다.

    Args:
        a: 첫 번째 숫자
        b: 두 번째 숫자
    """
    return a + b


print("add_numbers 도구가 등록되었습니다.")


## §5. 세 번째 도구 — 면적 계산

직사각형 면적을 계산합니다. 단위(unit) 인자가 추가되어 다양한 단위로 결과를 반환할 수 있습니다.

In [ ]:
# 도구 3: 면적 계산
@mcp.tool()
def calculate_area(width: float, height: float, unit: str = "m") -> str:
    """직사각형 면적을 계산합니다.

    Args:
        width: 너비
        height: 높이
        unit: 단위 - 예: m, mm, cm (기본값: m)
    """
    area = width * height
    return f"{area:.2f} {unit}²"


print("calculate_area 도구가 등록되었습니다.")


> ☑ **체크포인트 2**: 3개 도구가 모두 등록되었나요?
>
> 위의 3개 셀을 실행하면 각각 `... 도구가 등록되었습니다.`가 출력되어야 합니다.
> 동일한 도구를 두 번 등록하면 경고가 나올 수 있으니, 셀을 여러 번 실행한 경우 커널을 재시작하세요.


## §6. 자동 스키마 검증

FastMCP가 함수 시그니처에서 어떻게 JSON Schema를 자동 생성했는지 확인합니다.

In [ ]:
# 등록된 도구 목록과 자동 생성된 스키마 출력
import asyncio

async def list_tools_demo():
    """등록된 도구 목록과 스키마를 출력합니다."""
    tools = await mcp.list_tools()
    print(f"등록된 도구 수: {len(tools)}\n")
    for tool in tools:
        print(f"--- {tool.name} ---")
        print(f"  설명: {tool.description}")
        print(f"  스키마:")
        schema_str = json.dumps(tool.inputSchema, indent=4, ensure_ascii=False)
        # 들여쓰기를 위해 각 줄 앞에 두 칸 추가
        print("  " + schema_str.replace("\n", "\n  "))
        print()

await list_tools_demo()


## §7. 도구 호출 테스트

서버에 등록된 도구를 프로그래밍 방식으로 직접 호출하여 동작을 확인합니다.

> 💡 보통 도구 호출은 LLM이 자동으로 하지만, 여기서는 **개발자가 직접 호출**하여 도구가 정상 작동하는지 검증합니다.

In [ ]:
# 3개 도구를 직접 호출하여 검증
async def test_all_tools():
    # 도구 1: 시간 호출
    r1 = await mcp.call_tool("get_current_time", {"format": "%Y-%m-%d %H:%M"})
    print(f"현재 시간: {r1}")

    # 도구 2: 덧셈
    r2 = await mcp.call_tool("add_numbers", {"a": 3.14, "b": 2.71})
    print(f"3.14 + 2.71 = {r2}")

    # 도구 3: 면적 (300x600 mm)
    r3 = await mcp.call_tool("calculate_area", {"width": 300, "height": 600, "unit": "mm"})
    print(f"면적 (300x600mm): {r3}")

await test_all_tools()


> ☑ **체크포인트 3**: 세 도구 모두 결과를 반환했나요?
>
> 예상 출력:
> ```
> 현재 시간: ...현재 시각...
> 3.14 + 2.71 = ...5.85...
> 면적 (300x600mm): 180000.00 mm²
> ```
> 결과 형식이 약간 달라도(예: `[TextContent(...)]` 래핑) 정상입니다. SDK 버전에 따라 출력 구조가 다를 수 있습니다.


## §8. server.py 파일로 저장

지금까지 노트북에서 정의한 서버를 **독립 파일**(`tutorial_server.py`)로 저장합니다. 이 파일은 다음 노트북(T02 Inspector, T03 Client)에서 사용합니다.

> ⚠️ **튜토리얼 전용 파일명**: Skilljar 원본은 `server.py`를 쓰지만, 본 튜토리얼은 `tutorial_server.py`로 저장하여 충돌을 피합니다.

In [ ]:
# 위에서 정의한 서버 코드를 파일로 저장
server_code = """from mcp.server.fastmcp import FastMCP
from datetime import datetime

mcp = FastMCP(\"My First MCP Server\")


@mcp.tool()
def get_current_time(format: str = \"%Y-%m-%d %H:%M:%S\") -> str:
    \"\"\"현재 날짜와 시간을 반환합니다.

    Args:
        format: 날짜/시간 형식 (Python strftime 포맷)
    \"\"\"
    return datetime.now().strftime(format)


@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    \"\"\"두 숫자를 더합니다.

    Args:
        a: 첫 번째 숫자
        b: 두 번째 숫자
    \"\"\"
    return a + b


@mcp.tool()
def calculate_area(width: float, height: float, unit: str = \"m\") -> str:
    \"\"\"직사각형 면적을 계산합니다.

    Args:
        width: 너비
        height: 높이
        unit: 단위 - 예: m, mm, cm (기본값: m)
    \"\"\"
    area = width * height
    return f\"{area:.2f} {unit}²\"


if __name__ == \"__main__\":
    mcp.run()
"""

with open("tutorial_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

import os
print(f"OK - tutorial_server.py 저장 완료")
print(f"  절대 경로: {os.path.abspath('tutorial_server.py')}")
print(f"  파일 크기: {os.path.getsize('tutorial_server.py')} bytes")


## §9. 트러블슈팅 — 자주 만나는 에러 5가지

| # | 에러 메시지 | 원인 | 해결 |
|---|-----|------|------|
| 1 | `ModuleNotFoundError: No module named 'mcp'` | MCP SDK 미설치 | `pip install "mcp[cli]"` |
| 2 | `SyntaxError: 'await' outside function` | Jupyter 6.x | Jupyter 7.x 업그레이드 또는 `import nest_asyncio; nest_asyncio.apply()` |
| 3 | `AttributeError: 'FastMCP' object has no attribute 'tool'` | 잘못된 import | `from mcp.server.fastmcp import FastMCP` (server.fastmcp 위치 확인) |
| 4 | `pydantic.errors.PydanticUserError` | 타입 힌트 누락 | 함수 인자에 모두 타입 어노테이션 추가 (`a: float`) |
| 5 | `port 6277 already in use` | Inspector 포트 충돌 | T02에서 다룸 — `lsof -i :6277` + kill |

> 💡 **추가 팁**: 셀을 여러 번 실행하면 같은 도구가 중복 등록되어 경고가 발생할 수 있습니다. 이 경우 **Kernel → Restart**로 커널을 재시작하세요.


## §10. 다음 단계

✅ T01 완료! 이제 다음을 할 수 있습니다:
- FastMCP로 서버 생성
- `@mcp.tool()`로 도구 등록 (자동 스키마)
- 도구 직접 호출 테스트
- `tutorial_server.py` 파일 보유

➡️ **다음 노트북**: [`T02_inspector_walkthrough.ipynb`](T02_inspector_walkthrough.ipynb)에서 **MCP Inspector**(브라우저 UI)로 LLM 없이도 도구를 시각적으로 검증하는 법을 배웁니다.

> 📚 **추가 학습**:
> - `skilljar/S6_01_mcp_server.ipynb` — Skilljar 원본 (DocumentMCP 도메인)
> - `Week_07.md §1.4` — 도구 정의 심화
